In [1]:
import pandas as pd
import os

In [2]:
CSV_FILE_PATH = "../dataset.csv"
OUTPUT_DIR = "2_relational_model/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
df = pd.read_csv(CSV_FILE_PATH, delimiter=',')
print("CSV size before:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

CSV size before: (114000, 21)

Column names:
['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

First 3 rows:


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,...,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,...,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,...,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


In [ ]:
df = df.drop(columns=[col for col in df.columns if 'unnamed' in col.lower()])
print("Columns after drop:", df.columns.tolist())

Columns after drop: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


In [5]:
print("Null vrijednosti po stupcu:")
print(df.isnull().sum())
print(f"\nUkupno redaka s null vrijednostima: {df.isnull().any(axis=1).sum()}")

Null vrijednosti po stupcu:
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

Ukupno redaka s null vrijednostima: 1


In [6]:
df = df.dropna()
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("CSV size after dropna:", df.shape)

CSV size after dropna: (113999, 20)


In [7]:
# Konverzija 'explicit' u bool ako nije
df['explicit'] = df['explicit'].astype(bool)
print("Tip stupca 'explicit':", df['explicit'].dtype)
print(df['explicit'].value_counts())

Tip stupca 'explicit': bool
explicit
False    104252
True       9747
Name: count, dtype: int64


In [8]:
# Potpuni duplikati (identičan cijeli redak)
full_dupes = df.duplicated().sum()
print(f"Potpuni duplikati: {full_dupes}")

# Duplikati po track_idu (ista pjesma, različiti žanrovi - normalno u ovom datasetu)
track_id_dupes = df.duplicated(subset=['track_id']).sum()
print(f"Duplikati po track_id (ista pjesma u više žanrova): {track_id_dupes}")
print(f"Ukupno jedinstvenih track_id-eva: {df['track_id'].nunique()}")

Potpuni duplikati: 450
Duplikati po track_id (ista pjesma u više žanrova): 24259
Ukupno jedinstvenih track_id-eva: 89740


In [9]:
# Briše samo potpune duplikate, zadržava track_id duplikate jer jedan track može imati više žanrova
df = df.drop_duplicates()
print("CSV size nakon uklanjanja potpunih duplikata:", df.shape)

CSV size nakon uklanjanja potpunih duplikata: (113549, 20)


In [ ]:
# Provjera višestrukih izvođača (razdvojenih sa ';')
multi_artist = df['artists'].str.contains(';').sum()
print(f"Pjesme s više izvođača: {multi_artist}")
print("\nPrimjeri višestrukih: ")
print(df[df['artists'].str.contains(';')]['artists'].head(5).values)

Pjesme s više izvođača: 29893

Primjeri višestrukih izvođača:
['Ingrid Michaelson;ZAYN' 'A Great Big World;Christina Aguilera'
 'Jason Mraz;Colbie Caillat' 'Chord Overstreet;Deepend'
 'Andrew Foy;Renee Foy']


In [11]:
print(f"Broj jedinstvenih žanrova: {df['track_genre'].nunique()}")
print(f"\nRaspon popularnosti: {df['popularity'].min()} - {df['popularity'].max()}")
print(f"Prosječna popularnost: {df['popularity'].mean():.2f}")
print("\nTop 5 žanrova po broju pjesama:")
print(df['track_genre'].value_counts().head())

Broj jedinstvenih žanrova: 114

Raspon popularnosti: 0 - 100
Prosječna popularnost: 33.32

Top 5 žanrova po broju pjesama:
track_genre
acoustic       1000
emo            1000
rock-n-roll    1000
reggaeton      1000
disco          1000
Name: count, dtype: int64


In [12]:
df20 = df.sample(frac=0.2, random_state=42)
df80 = df.drop(df20.index)
print("Veličina 80% skupa:", df80.shape)
print("Veličina 20% skupa:", df20.shape)

Veličina 80% skupa: (90839, 20)
Veličina 20% skupa: (22710, 20)


In [13]:
df80.to_csv(f"{OUTPUT_DIR}/spotify_tracks_PROCESSED.csv", index=False)
df20.to_csv(f"{OUTPUT_DIR}/spotify_tracks_PROCESSED_20.csv", index=False)
print("Datoteke uspješno spremljene.")
print(f"  → {OUTPUT_DIR}/spotify_tracks_PROCESSED.csv")
print(f"  → {OUTPUT_DIR}/spotify_tracks_PROCESSED_20.csv")

Datoteke uspješno spremljene.
  → 2_relational_model/processed/spotify_tracks_PROCESSED.csv
  → 2_relational_model/processed/spotify_tracks_PROCESSED_20.csv


In [14]:
print("=== FINALNI PREGLED (80% skup) ===")
print(df80.dtypes)
print("\nPrvih 5 redaka:")
df80.head()

=== FINALNI PREGLED (80% skup) ===
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

Prvih 5 redaka:


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,0.481,6,-8.807,1,0.1050,0.2890,0.000000,0.1890,0.666,98.017,4,acoustic
